<a href="https://colab.research.google.com/github/yunqilu/MANDARIN-Convert-SPiN-Lists-to-WAV-and-create-transcript-file/blob/main/MANDARIN_Convert_SPiN_Lists_to_WAV_and_create_transcript_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code to assemble the alternate speech in noise files into a form that Kent can read into the SPIN database and serve via the web test.

We need the following columns in the CSV:

*   active (0 or 1): if this example should be used (used to disable some of the QS lists)
*   lang (test name): Let's use this for the test name, XX-LANG, where XX is the test name, and LANG is the language.  Right now EN means QuickSIN in English
*   trial_number (integer): The SPIN test list number (1-12 for QuickSIN)
*   level_number (integer): The sentence within the list
*   snr: The signal to noise ratio, probably only matters for QuickSIN
*   filename: Where to find the audio. Just the last part of the path
*   answer: the desired answer words, comma separated.  These are all the target words.




The original audio files can be any where, but are typically in this folder: drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials

After processing by this code, they are copied to this folder: drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio.

In [1]:
import glob
import os
import re

from typing import List, Tuple

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
!ls "/content/drive"

MyDrive  Shareddrives


In [4]:
!ls "/content/drive/Shareddrives"

 StanfordAudiology  'Stanford Solar Car Project Drive'


In [5]:
basedir = 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials'
!ls '{basedir}/Web_Audio/azbio_mandarin_quiet' | head

ls: cannot access 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio/azbio_mandarin_quiet': No such file or directory


In [6]:
basedir = 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials'
!ls '{basedir}/Web_Audio/azbio_mandarin_quiet' | head

ls: cannot access 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio/azbio_mandarin_quiet': No such file or directory


In [ ]:
!rm -rf /tmp/*.wav /tmp/*.csv

web_audio_dir = os.path.join(basedir, 'Web_Audio')
# web_audio_dir = '/tmp'

In [ ]:
!ls "{web_audio_dir}"

ls: cannot access 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio': No such file or directory


In [ ]:
nu6_dir = os.path.join(basedir, 'NU6_List1')
!ls '{nu6_dir}'

ls: cannot access 'drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/NU6_List1': No such file or directory


In [ ]:
class SpinData(object):
  def __init__(self,
               sound_directory: str,
               output_directory: str, suffix: str = 'wav'):
    """
    Args:
      sound_directory: Where to find the original sounds
      output_directory: The base directory for all sound files.  The files from
        this type of spin data will go into the test_name folder within this
        folder.
      suffix: the last part of the original (segmented) audio file names.
    """
    try:
      self.test_name
    except NameError:
      raise ValueError('Must specify a test_name in the subclass')
    self.sound_directory = sound_directory
    self.metadata_dir =  os.path.join(output_directory, 'metadata')
    os.makedirs(self.metadata_dir, exist_ok=True)
    self.transcript_filename = os.path.join(self.metadata_dir,
                                            self.test_name + '_transcript.csv')
    self.output_directory = os.path.join(output_directory, self.test_name)
    os.makedirs(self.output_directory, exist_ok=True)
    self.suffix = suffix

  def all_audio_files(self) ->  List[str]:
    full_glob = os.path.join(self.sound_directory, '*.'+self.suffix)
    all_files = glob.glob(full_glob)
    # all_files = [f.split('/')[-1].replace('.aif', '') for f in all_files]
    return all_files

  def convert_to_wave(self, input, output) -> None:
    !ffmpeg -hide_banner -loglevel error -y -i '{input}' '{output}'

  def keep_file(self, basename: str) -> bool:
    return True

  def adjust_output_filename(self, filename: str) -> str:
    if filename.endswith('.aif'):
      filename = filename.replace('.aif', '.wav')
    return filename

  def write_csv_transcript(self, basename: str, fp):
    test_name, list_num, trial_num, transcript = self.parse(basename)
    basename = basename.split('/')[-1]
    fp.write(f'1, en, {list_num}, {trial_num}, {self.test_name}/{basename}, {transcript}\n')

  def process(self, max_files=0) -> None:
    all_audio_files = self.all_audio_files()
    assert len(all_audio_files) > 0, f'No audio files found in {self.sound_directory}'
    print(f'Found {len(all_audio_files)} audio files in {self.sound_directory}')
    with open(self.transcript_filename, 'w') as fp:
      file_count = 0
      for input_filename in sorted(all_audio_files):
        basename = input_filename.split('/')[-1]
        if not self.keep_file(basename):
          continue
        output_filename = self.adjust_output_filename(basename)
        output_filename = os.path.join(self.output_directory, output_filename)

        print(f'Processing {basename}')
        self.convert_to_wave(input_filename, output_filename)
        self.write_csv_transcript(basename, fp)
        file_count += 1
        if max_files > 0 and file_count >= max_files:
          break

In [ ]:
sorted([4,2,4,1])

[1, 2, 4, 4]

# AzBio Mandarin Transcripts

Add other lists here. Remove all optional words (the ones in parentheses)

In [ ]:
AzBioMandarinList = {}

AzBioMandarinList[1] = """
1 火车站 有车 可以 直达 机场 5
2 肺癌的 患病 率 依然 在 升高 6
3 现在 很多 楼房 都 建有 地下室 6
4 明天 最后 期限 得 加班了 5
5 这家的 川菜 做得 很 地道 5
6 主任 安排 低 年级 学生 打扫 操场 7
7 那张 茶几的 设计 是 不 规则的 6
8 我 很高兴 你们 俩 能 过来 参观 7
9 这 餐馆 外地 有 分店 5
10 电脑 是他 花费 最 多的 一样 东西 7
11 孩子 眼巴巴地 望着 想要的 玩具 5
12 我 知道 一家 很棒的 小店 一起 去吧 7
13 你 比较 适合 圆领的 衬衫 5
14 傍晚的 时候 会 有 人 来 换班 7
15 爸爸 晚上 常常 陪 孩子 睡觉 6
16 多数 人 一辈子 都 没有 你 这样的 福气 8
17 你的 那些 故事 我 都 听说 过了 7
18 长期 饮用 酒精 会 使 感觉 迟钝 7
19 怎么 用 电话 打 国际 长途 6
20 他 今年 五十五 岁 看 却 很 年轻 8
"""


AzBioMandarinList[2] = """
1 这本 旧 书 是 从 别人 那里 淘来的 8
2 孩子们 戴着 动物 面具 游戏 5
3 我 站得 很 远 也能 听见 你的 声音 8
4 现在 很少 有人 会 寄 挂号信 了吧 7
5 我们 统一 一个 碰头的 地点 5
6 他 企图 把 锁 撬开 但 失败了 7
7 这周 轮到 谁 打扫 厕所了 5
8 需要 多长 时间 才能 将 电池 充满 7
9 我 十天 之内 肯定 给你 答复 6
10 医生 告诉 我 靠 吃药 不能 去 病根 8
11 他 英语 说得 一点 也不 流利 6
12 我 穿 太 多了 行动 不便 6
13 晚上 七点 我们 广场 集合 5
14 海豚的 智商 比 很多 动物 都 高 7
15 白酒 里面 不 应该 有 醋的 味道 7
16 我 希望 能 将 这个 习惯 坚持 下来 8
17 这番 话 都是 我的 经验 总结 啊 7
18 他 把 剩下的 午饭 喂 狗了 6
19 这个 房间 给 人的 感觉 有点 冷清 7
20 不是 所有 人的 工作 都与 专业 有关 7
"""

AzBioMandarinList[3] = """
1 你 别 让 家人 跟着 着急 6
2 选举 最终 按照 预定的 时间 开始 6
3 路边儿 小吃 摊儿的 东西 真 让人 不放心 7
4 怎样 才能 练 好 普通 话 6
5 这位 学者的 作品 在市场上 找不到 5
6 男人 去世 后 没有 留下 遗嘱 6
7 这 月 十六号 是 星期 几？ 6
8 上次 我 超速 没有 被 发现 6
9 不要 总是 在 大晚上 看 恐怖 片 7
10 工人们 把 石头 垒成了 一道 堤坝 6
11 你 尝尝 这 季节 的螃蟹 可 肥了 7
12 这 画报 是 我 从 朋友 手里 换来的 8
13 我 最 期待的 节目 是 魔术 6
14 我 很快 就 重新 开始 训练 6
15 出租车 司机 对 我 非常 不 礼貌 7
16 你 能 在 便利店 顺便 买 一瓶 牛奶吗 8
17 我 觉得 这个 教授的 观点 很 对 7
18 这片 土地的 作物 生长 良好 5
19 开车 千万 别 和 我 说话 6
20 虽然 城市 很小 但是 经济 发达 6
"""

AzBioMandarinList[4] = """
1 公司 派 我 作为 代表 去 谈判 7
2 观众们 自觉地 按 秩序 入场 5
3 我 每年 冬天 都 养 水仙花 6
4 不敢 想象 他 该 怎么 收拾 我 7
5 报名的 截止 日期 推后 两天 5
6 这套 书的 精装本 非常 抢手 5
7 我 照着 说明书 做了 四个 样品 6
8 我 早起 第一件 事情 就是 喝水 6
9 我 感觉 今天 什么 都没 干 6
10 他有 十年的 高层 管理 经验 5
11 子女们 都 搬 出去 自己 住了 6
12 姑娘们 爽快地 答应了 我的 请求 5
13 我 琢磨着 这 里面 有 问题 6
14 冰箱里的 几个 桃子 好像 都 长毛了 6
15 他 从 罐子里 拿出 一些 饼干 给 小狗 8
16 回去的 时候 别 忘了 拿 你的 东西 7
17 他 这么 说 是 为了 安慰 我 7
18 现在 就 答应 这个 条件 有点 轻率 7
19 他们 购买了 一组 很 高档的 沙发 6
20 我们 跟 这个 卖 衣服的 结账吧 6
"""

AzBioMandarinList[5] = """
1 你 喜欢 日本 菜 还是 泰国？ 6
2 编 一部 书 需要 积累 多年的 资料 7
3 咱们 需要 找 人 来 修理 下水道 7
4 家里的 百叶窗 已经 不 牢固了 5
5 你 能 想起 事情的 细节吗 5
6 他 口袋里的 硬币 是 他的 幸运物 6
7 晚饭 之后 他 显得 很 不 开心 7
8 买 之前 要不要 先 测量 一下 尺寸 7
9 这 好处 我可 不能 白 拿 6
10 银行 附近 正在 建造 一栋 新的 公寓 楼 8
11 颜料 瓶子的 盖子 被 他 弄丢了 6
12 一个 小女孩儿 独自 坐在 公园里 5
13 这段 话 放在 这里 是 多余的 6
14 我 想 查一下 过去 一周的 交易 记录 7
15 我 喜欢 吃 妈妈 做的 饺子 6
16 我 得 仔细 阅读 所有 材料 6
17 您 看起来 真 不像 本地 人 6
18 从来 没有 听过 如此 荒唐的 理由 6
19 这个 测试的 结果 很 令人 失望 6
20 这种 材质 完全 不 符合 标准 6
"""

AzBioMandarinList[6] = """
1 他 总 喜欢 拿 我 恶作剧 6
2 瘟疫 可能 与 去年的 水灾 有关 6
3 这堆 书 到 现在 才 卖 出去 一半儿 8
4 这本 小 册子 上 有 各个 部门的 号码 8
5 你的 判断 从来 没有 对 过 6
6 请 把 那些 杂志 送 我的 办公室 7
7 感冒 药的 疗效 不 怎么样啊 5
8 刚才 走神儿 没 听见 你 叫 我 7
9 新建的 火车站 有 两个 寄存处 5
10 她在 滑雪 比赛 获得了 一块 铜牌 6
11 这条 走廊 一直 通 卧室 5
12 天黑 以后 公园里 有一个 大型 晚会 6
13 这句 话 我 都 憋了 好久了 6
14 两位 记者 在 报纸 互相 指责 6
15 乱翻 别人的 东西 很不 礼貌 5
16 真 没 想 到 会在 这儿 碰到 你 8
17 可以 帮 我 把 文件 送 办公楼吗 7
18 有点 事 我 得 听听 你的 意见 7
19 他 正在 休息 你 别 大喊 叫 的 8
20 经过 仔细 排查 他们 才 发现 事故的 原因 8
"""

AzBioMandarinList[7] = """
1 第一次 上台 发言 好 紧张啊 5
2 数字 和 运气 不可能 有 什么 关系 7
3 下 礼拜 没有 一天 好 天气 6
4 说得 轻松 我 可 没有 那么 大 把握 8
5 这篇 报道 不得 涉及 隐私 5
6 安全 起见 这条 路 已经 封了 6
7 秋天 去 北京 正好 碰上 旅游 高峰 7
8 他 母亲的 健康 状况 迅速 恶化 6
9 奶奶 每天 都 会 公园 晨练 6
10 估计 他 这会儿 正在 挨 骂呢 6
11 咱们 来 看 一下 景点儿的 介绍吧 6
12 我 工作 这么 久 从 没 抱怨 过 8
13 今年 市场上的 橘子 比 往年 便宜 6
14 水面 上 有 几片 枯萎的 荷叶 6
15 记者 无法 确定 这次 事件的 起因 6
16 你 弟弟 身体 可 比 我 强壮 多了 8
17 他 被 突然的 消息 惊呆了 5
18 我的 脸 让 蜜蜂 蛰了 一下 6
19 他 好不 容易 才 租 到了 房子 7
20 他 偷偷 从 被子 下面 探出 头 7
"""

AzBioMandarinList[8] = """
1 房间 里 手机 总是 没有 信号 6
2 我 刚 学了 一个 新 把戏 6
3 他们 打算 乘 船 游览 两岸 风光 7
4 不要 担心 他 是 做事儿 沉稳的 人 7
5 我 听说 公园的 茶花 开了 5
6 老板 今天 来 检查 雇员的 工作 情况 7
7 我 把 这个 换成 更 大的 尺寸 7
8 他们的 建议 和 我们的 有 相似 之处 7
9 最近的 工作 强度 让人 觉得 压抑 6
10 不知道 这 绳子 能 否 承受 我的 重量 8
11 明天的 音乐 会 会 有 很多 艺术 家 8
12 今年 夏天 我们 换个 新 纱窗吧 6
13 前面 路口 左 拐 是 什么 地方 7
14 这位 演员 被 授予了 最佳 女 主角 7
15 他 没有 察觉 老板 生气了 5
16 他 身上 伤 所以 不能 当 飞行员 7
17 明天 晚上 我们 海边 露营 5
18 我 能 换到 其他 座位 上去吗 6
19 今年 我 想 报名 参加 马拉松 6
20 他们 两个 本来 就 不是 同 一类 人 8
"""

AzBioMandarinList[9] = """
1 最近 爷爷的 血压 不太 稳定 5
2 厨房 用具 都 不锈钢 做 5
3 技校 注重 是 学生的 实践 能力 6
4 接替 我 工作的 人 暂时 还 没 到位 8
5 我的 证件 应该 还 没有 过期吧 6
6 船长 命令 船员们 把 重物 搬 甲板上 7
7 叔叔 酒 喝 多 就 喜欢 乱 说话 8
8 他 工作的 第一天 就 被 解雇了 6
9 你 和 谁 去 游泳池 玩的 6
10 单 从 外表 很难 判定 一个 人的 性格 8
11 孩子 做 什么 都 喜欢 模仿 父母 7
12 这双 鞋子 不 适合 用来 跑步 6
13 这些 杯子 都 我 市场 买的 6
14 你 穿上 职业 装 显得 挺 精神 7
15 我 把 爷爷的 手表 弄 坏了 6
16 我的 体力 还可以 继续 坚持 5
17 我 对 现在的 生活 很 满足 6
18 办公 用品 需要 发票 才能 报销 6
19 我 比较 中意 这块儿 布的 图案 6
20 你 要的 文件 我 已经 帮你 打印 好了 8
"""

AzBioMandarinList[10] = """
1 起来 活动 活动 就 不 冷了 6
2 我 想 车上 安装 一个 小型 电视 7
3 光 他俩的 衣服 就 把 柜子 塞 满了 8
4 我 保证 你 这次 旅行 能 尽兴 7
5 你 现在 还 坚持 集邮吗 5
6 别 总 把 你的 迟到 归结于 交通 7
7 刚 成家的 时候 我 连 饭 不会 做 8
8 这家 餐馆的 菜 不加 味精 5
9 这件 事 他 已经 跟我 打 保票了 7
10 他 想 搭 便车 去 看望 母亲 7
11 这套 旧 家具 能 值 很多 钱 7
12 我 中途 花了 一个 小时 挑选 明信片 7
13 看 这 天气 恐怕 飞 不了 了 7
14 到 现在 我 还 记得 这句 台词 7
15 场地 太 湿了 没 办法 踢球 6
16 门 架子 上的 葡萄 藤 完全 枯萎了 7
17 你 没有 理解 这种 计算 方法 6
18 这场 雨 有助于 扑灭 大火 5
19 我 做的 饭菜 还是 上得了 台面的 6
20 前 两天 你 不是 已经 和 我 讨论 过了吗 9
"""

AzBioMandarinList[11] = """
1 可以 帮 我 挑 两棵 白菜吗 6
2 这个 停车场 仅 供 内部 员工 使用 7
3 老奶奶 给 小朋友 一本 故事书 5
4 快点儿 我们 要错过 那部 片子的 开头了 6
5 麻烦 您 帮 我 拿 那件 毛衣 试试 8
6 渔民 把 小船 划 到了 湖 中心 7
7 别把 茶 泡得 太 浓 我 喝 不惯 8
8 一连 几天 都 没有 一个 顾客 上门儿 7
9 我们 还能 找到 合适的 酒店 办 婚礼吗 7
10 这项 研究 没 什么 实际 用途 6
11 公交 司机 对 乘客的 态度 很 亲切 7
12 我 不 介意 家里 没有 固定 电话 7
13 你 最好 写 一张 正式的 请假 条 7
14 农夫们 正在 果园 打 农药 5
15 他 对 事情的 描述 不 值得 相信 7
16 这 对于 我 确实 是 一个 绝佳的 机会 8
17 您 可以 帮 我 买 一把 香蕉 回来吗 8
18 传统的 年夜饭 必须 有 鱼 5
19 每年 会 有 很多的 游客 爬 长城 7
20 我的 家乡 是 一个 不 起眼的 小山村 7
"""

AzBioMandarinList[12] = """
1 你 究竟 有 没有 拿 我 当 好 朋友 9
2 梅雨 季节 需要 持续 两三个 月 6
3 我 把 棉衣 捐献 给了 灾民 6
4 我 要 看看 其他 摊子 都 在 卖 什么 9
5 中午 我 要 去 医院 照顾 妈妈 7
6 母亲 往 汤里 加了 一点 酱油 6
7 每月的 支出 不要 超过 工资的 一半 6
8 暂时 把 画框 放 地板上吧 5
9 你 打算 怎么 用 今年的 奖学金 6
10 这座 城市 临近 全国 闻名的 小 商品 市场 8
11 饭店 附近 没有 地方 停车了 5
12 可以 告诉 我 派出所 怎么 走吗 6
13 你 答应 这周 会 教 我 打 保龄球 8
14 每条 绳子 都 被 卷成 一捆儿 6
15 主任 同意 你 重新 制定 计划 6
16 夫妻 俩的 收入 加 起来 才能 维持 开支 8
17 我 好想 回到 大家 一起 上学的 日子啊 7
18 孩子 艰难地 从 地上 爬 起来 6
19 他 迷上了 电脑 游戏 不 爱 学习 7
20 如果 用 这种 方法 就 可以 解决 问题 8
"""

AzBioMandarinList[13] = """
1 我们 学校 来了 十个 新 老师 6
2 司机 和 行人 应该 相互 谦让 6
3 冬天的 时候 外出 清理 积雪 挺 费劲 7
4 家里 所有 人 都 聚到了 爷爷 身边 7
5 这个 方案 很 实际 很 合理 6
6 我 不 知道 书名 但是 还 记得 作者 8
7 山里 风 大 你 穿得 太 单薄了 7
8 大 部分的 花儿 最 多 开 一个 月 8
9 父母 一直 支持 我的 工作 5
10 农夫 整个 下午 都在 院子 晒 太阳 7
11 奶奶的 抽屉 有 几个 漂亮的 戒指 6
12 今天会 下雨 我 用不着 浇花儿了 5
13 他 给 职工 争取 到了 最大的 福利 7
14 我 认为 网络 不能 代替 书籍 6
15 孩子 在 六岁 前后 开始 换牙 6
16 要是 有 机会 我 还会 再 尝试 一次 8
17 还没 毕业 我 就要 开始 找 工作了 7
18 你 怎么 总 不分 场合 打断 别人 说话 8
19 我们 都 很 担心 爷爷的 身体 6
20 我 来的 时候 他 已经 在 这儿了 7
"""

AzBioMandarinList[14] = """
1 我 今天 接待了 一个 澳洲的 旅行 团 7
2 做 这项 工作 最好 了解 一些 背景 知识 8
3 道路 施工 造成了 好 几起 交通 意外 7
4 我的 头发 还是 像 平常 那样 剪 7
5 他 最近 迷上了 这部 新 电视剧 6
6 父亲 给 木头 家具 打上 蜡 6
7 现在的 失业 率 比 以前 高 多了 7
8 为什么 不 膝盖上 敷个 热 毛巾 6
9 我 已经 帮 你们 找好 旅馆了 6
10 一 到 目的地 立刻 给 我 回 电话 8
11 根本 没有 人 在意 你的 收入 6
12 我 看 这对 耳环 不是 非常 适合 你 8
13 我们 不得不 又 赶回 公司 一趟 6
14 在 现 阶段 出现了 一些 新 问题 7
15 他 收到了 公司 开业的 邀请 函 6
16 采取 这项 措施 可以 降低 成本 6
17 真 后悔 今天 出门 没 带 伞 7
18 没有 处方 你 不能 买 那种 药 7
19 这道 点心 真的 太 美味了 5
20 我 知道 你 不会 泄漏 这个 消息的 7
"""

AzBioMandarinList[15] = """
1 这篇 评论的 指向性 太 强 5
2 他的 眼睛 死死 盯着 变化的 数字 6
3 我 必须 出发 去 汽车 站了 6
4 我 和 好友 学校 附近 合租 一套 房子 8
5 他们 在 前面 的路口 等 我 6
6 大量的 需求 使 猪肉 价格 上涨 6
7 我 很 生气 他 总是 欺负 人 7
8 我们 上班 时 就 把 女儿 送 托儿所 8
9 我 看 你 什么 成绩 交差 6
10 下次 别 指望 我 能够 帮 你了 7
11 孩子 正 蹲 在 地上 看 蚂蚁 7
12 提前 几个 月 可以 买到 便宜的 机票 7
13 这些 东西 您 先 拿着 应急 6
14 路旁的 广告 牌儿 有时 会 影响 视线 7
15 昨天 半夜 发生了 一起 偷盗 案件 6
16 真 不明白， 他 当初 为什么 要 做 这份 工作 9
17 他们 三个 都 我 初中 时的 同学 7
18 影片 里的 一些 镜头 被 删掉了 6
19 这部 电影 是 根据 真 事儿 改编的 7
20 这个 星期的 饭钱， 你 都 花在 哪儿了 7
"""

Format lists

In [ ]:
AzBio_Mandarin_Transcripts = {}

for list_num, list_text in AzBioMandarinList.items():
    lines = list_text.strip().split('\n')

    for line in lines:
        trial_num, transcript = line.split(' ', 1)

        # 删除最后面的数字
        transcript = ' '.join(transcript.split()[:-1])

        AzBio_Mandarin_Transcripts[(list_num, int(trial_num))] = transcript

In [ ]:
for list_num, text in AzBioMandarinList.items():
    new_lines = []

    for line in text.strip().split('\n'):
        parts = line.split()

        # 去掉最后一个数字
        new_lines.append(' '.join(parts[:-1]))

    AzBioMandarinList[list_num] = new_lines

AzBioMandarinList

{1: ['1 火车站 有车 可以 直达 机场',
  '2 肺癌的 患病 率 依然 在 升高',
  '3 现在 很多 楼房 都 建有 地下室',
  '4 明天 最后 期限 得 加班了',
  '5 这家的 川菜 做得 很 地道',
  '6 主任 安排 低 年级 学生 打扫 操场',
  '7 那张 茶几的 设计 是 不 规则的',
  '8 我 很高兴 你们 俩 能 过来 参观',
  '9 这 餐馆 外地 有 分店',
  '10 电脑 是他 花费 最 多的 一样 东西',
  '11 孩子 眼巴巴地 望着 想要的 玩具',
  '12 我 知道 一家 很棒的 小店 一起 去吧',
  '13 你 比较 适合 圆领的 衬衫',
  '14 傍晚的 时候 会 有 人 来 换班',
  '15 爸爸 晚上 常常 陪 孩子 睡觉',
  '16 多数 人 一辈子 都 没有 你 这样的 福气',
  '17 你的 那些 故事 我 都 听说 过了',
  '18 长期 饮用 酒精 会 使 感觉 迟钝',
  '19 怎么 用 电话 打 国际 长途',
  '20 他 今年 五十五 岁 看 却 很 年轻'],
 2: ['1 这本 旧 书 是 从 别人 那里 淘来的',
  '2 孩子们 戴着 动物 面具 游戏',
  '3 我 站得 很 远 也能 听见 你的 声音',
  '4 现在 很少 有人 会 寄 挂号信 了吧',
  '5 我们 统一 一个 碰头的 地点',
  '6 他 企图 把 锁 撬开 但 失败了',
  '7 这周 轮到 谁 打扫 厕所了',
  '8 需要 多长 时间 才能 将 电池 充满',
  '9 我 十天 之内 肯定 给你 答复',
  '10 医生 告诉 我 靠 吃药 不能 去 病根',
  '11 他 英语 说得 一点 也不 流利',
  '12 我 穿 太 多了 行动 不便',
  '13 晚上 七点 我们 广场 集合',
  '14 海豚的 智商 比 很多 动物 都 高',
  '15 白酒 里面 不 应该 有 醋的 味道',
  '16 我 希望 能 将 这个 习惯 坚持 下来',
  '17 这番 话 都是 我的 经验 总结 啊',
  '18 他 把 剩下的 午饭 喂 狗了',
  '19 这个 房间 给 人的 感觉 有点 冷

# AzBio Mandarin


Note - audio file naming convention is slightly different than what Yunqi and Emily first did. (AzBio_Mandarin_10dBSNR_List1_Sentence-01.wav). Variation name has to do with the signal to noise ratio (SNR).

## Starter code for other test formats (words, sentences with variable SNR, etc) can be found here: https://colab.research.google.com/drive/1txDoUsXjpCYttYnLdup7ZpaZ15A1tL0p?usp=sharing
. I am happy to go over it with you as we add more tests in Mandarin so that the naming conventions correspond to the code!

In [ ]:
class SpinAzBioMandarin(SpinData):
  def __init__(self,
               sound_directory: str,
               output_directory: str,
               test_name: str = 'azbio_mandarin', *args, **kwords):
    self.test_name = test_name
    super().__init__(sound_directory=sound_directory,
                     output_directory=output_directory)

  def write_csv_transcript(self, basename: str, fp):
    test_name, list_num, trial_num, transcript = self.parse(basename)
    basename = basename.split('/')[-1]
    fp.write(f'1, zh, {list_num}, {trial_num}, {self.test_name}/{basename}, {transcript}\n')

  def parse(self, filename) -> Tuple[str, int, int, str]:
    # AzBio_Mandarin_10dBSNR_List1_Sentence-01.wav
    (test_name, language_name, variation_name, list_name, word_name) = filename.split('_')
    test_name = test_name + ('_') + language_name # put language back in
    assert list_name.startswith('List')
    list_num = int(list_name.replace('List', ''))
    assert word_name.startswith('Sentence-')
    trial_num = int(word_name.replace('Sentence-', '').replace('.wav', ''))
    transcript = AzBio_Mandarin_Transcripts[(list_num, trial_num)]
    return test_name, list_num, trial_num, transcript

In [ ]:
azbio_mandarin_list = SpinAzBioMandarin(os.path.join(basedir, 'AzBio_Mandarin/*List*/'),
                       web_audio_dir, suffix='wav')

azbio_mandarin_list.process(max_files=100000)

OSError: [Errno 95] Operation not supported: 'drive/Shareddrives/StanfordAudiology'

In [ ]:
!ls -R '{azbio_mandarin_dir}' | grep .wav

ls: cannot access '{azbio_mandarin_dir}': No such file or directory


In [ ]:
# !ls "{azbio_dir}"
!wc '{azbio_mandarin_list.transcript_filename}'
!ls -lsa '{azbio_mandarin_list.output_directory}' | head
!ls -lsa '{azbio_mandarin_list.output_directory}'/*.wav | wc

  40  457 4640 drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio/metadata/azbio_mandarin_transcript.csv
total 83772
  691 -rw------- 1 root root   707442 Jun 21 20:50 AzBio_Mandarin_10dBSNR_List1_Sentence-01.wav
  691 -rw------- 1 root root   707442 Jun 21 20:06 AzBio_Mandarin_10dBSNR_List1_Sentence_01.wav
  691 -rw------- 1 root root   707442 Jun 21 20:10 AzBio_Mandarin_10dBSNR_List1_Sentence01.wav
  759 -rw------- 1 root root   776238 Jun 21 20:50 AzBio_Mandarin_10dBSNR_List1_Sentence-02.wav
  759 -rw------- 1 root root   776238 Jun 21 20:19 AzBio_Mandarin_10dBSNR_List1_Sentence_02.wav
  676 -rw------- 1 root root   691566 Jun 21 20:50 AzBio_Mandarin_10dBSNR_List1_Sentence-03.wav
  709 -rw------- 1 root root   725964 Jun 21 20:50 AzBio_Mandarin_10dBSNR_List1_Sentence-04.wav
  697 -rw------- 1 root root   713616 Jun 21 20:50 AzBio_Mandarin_10dBSNR_List1_Sentence-05.wav
  746 -rw------- 1 root root   763008 Jun 21 20:50 AzBio_Mand

# AzBio Mandarin Quiet


In [ ]:
class SpinAzBioMandarinQuiet(SpinData):
  def __init__(self,
               sound_directory: str,
               output_directory: str,
               test_name: str = 'azbio_mandarin_quiet', *args, **kwords):
    self.test_name = test_name
    super().__init__(sound_directory=sound_directory,
                     output_directory=output_directory)

  def write_csv_transcript(self, basename: str, fp):
    test_name, list_num, trial_num, transcript = self.parse(basename)
    basename = basename.split('/')[-1]
    fp.write(f'1, zh, {list_num}, {trial_num}, {self.test_name}/{basename}, {transcript}\n')

  def parse(self, filename) -> Tuple[str, int, int, str]:
    # AzBio_Mandarin_Quiet_List1_Sentence-01.wav
    (test_name, language_name, variation_name, list_name, word_name) = filename.split('_')
    test_name = test_name + ('_') + language_name # put language back in
    assert list_name.startswith('List')
    list_num = int(list_name.replace('List', ''))
    assert word_name.startswith('Sentence-')
    trial_num = int(word_name.replace('Sentence-', '').replace('.wav', ''))
    transcript = AzBio_Mandarin_Transcripts[(list_num, trial_num)]
    return test_name, list_num, trial_num, transcript

In [ ]:
azbio_mandarin_quiet_list = SpinAzBioMandarinQuiet(os.path.join(basedir, 'AzBio_Mandarin_Quiet/*List*/'),
                       web_audio_dir, suffix='wav')

azbio_mandarin_quiet_list.process(max_files=100000)

OSError: [Errno 95] Operation not supported: 'drive/Shareddrives/StanfordAudiology'

In [ ]:
!ls -R '{azbio_mandarin_quiet_dir}' | grep .wav

ls: cannot access '{azbio_mandarin_quiet_dir}': No such file or directory


In [ ]:
# !ls "{azbio_dir}"
!wc '{azbio_mandarin_quiet_list.transcript_filename}'
!ls -lsa '{azbio_mandarin_quiet_list.output_directory}' | head
!ls -lsa '{azbio_mandarin_quiet_list.output_directory}'/*.wav | wc

  40  457 4800 drive/Shareddrives/StanfordAudiology/Research - stimuli and programs/segmented speech materials/Web_Audio/metadata/azbio_mandarin_quiet_transcript.csv
total 29446
691 -rw------- 1 root root 707442 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-01.wav
691 -rw------- 1 root root 707442 Jun 21 20:27 AzBio_Mandarin_Quiet_List1_Sentence_01.wav
759 -rw------- 1 root root 776238 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-02.wav
676 -rw------- 1 root root 691566 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-03.wav
709 -rw------- 1 root root 725964 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-04.wav
697 -rw------- 1 root root 713616 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-05.wav
746 -rw------- 1 root root 763008 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-06.wav
680 -rw------- 1 root root 695976 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-07.wav
709 -rw------- 1 root root 725082 Jun 21 20:50 AzBio_Mandarin_Quiet_List1_Sentence-08.wav
     41     